In [ ]:
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting

# First part

In [ ]:
import numpy as np
from scipy.special import jv

INF = np.inf

hbar = 1
e = 1
#beta = INF  # Zero-temperature case: T = 0, beta -> inf
phi_quantum = 1 / 2

# Circuit parameters
omega_a = 25.338776456203686
omega_b = 2 * omega_a

phi_a = 0.11
phi_b = 0.204

E_J = 37.12 * 2 * np.pi 

# Environment
kappa_b = 4 / 10.4  # 1/ns from Leon's paper

omega_d = omega_b                  # Drive frequency
omega_p = 2 * omega_a - omega_d   # Pump frequency

epsilon_p = 0.1
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b  # Leon's paper
epsilon_d = 4 * g2

g = E_J * np.sin(epsilon_p) * phi_a**2 * phi_b
epsilon_d_ratio = 2
params = (epsilon_p, E_J, epsilon_d_ratio, kappa_b)

M = 100

A_0 = np.sqrt(epsilon_d / g)
B_0 = np.sqrt(2) * epsilon_d / kappa_b

In [ ]:
kappa_b < 4*np.sqrt(2)*g*A_0

In [ ]:
from scipy.integrate import solve_ivp
alpha = kappa_b / (2*np.sqrt(2)*g*A_0)
x0 = A_0

def rhs(x, f):
    phi = f[0]
    return [alpha + x*(1 - x/x0)/phi]

# choose branch of the local slope near x=0
c_plus  = 0.5*(alpha + np.sqrt(alpha**2 + 4))
c_minus = 0.5*(alpha - np.sqrt(alpha**2 + 4))

# pick one branch
c = c_minus

eps = 1e-6
x_init = eps
f_init = c * eps

x_fin = x0

sol = solve_ivp(
    rhs,
    t_span=(x_init, x_fin),
    y0=[f_init],
    dense_output=True,
    max_step=0.01,
    rtol=1e-8,
    atol=1e-10
)

print(sol.message)

In [ ]:
import matplotlib.pyplot as plt
xx = np.linspace(x_init, x_fin, 1000)
yy = sol.sol(xx)[0]

plt.plot(xx, yy)
plt.xlabel("x")
plt.ylabel(r"$\phi(x)$")
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

# parameters
c = np.sqrt(2) * g * A_0

# reduced vector field X = (A, PB)
def F(X):
    PA, B = X
    dPA =  - c * B
    dB = np.sqrt(2) * g * PA * (A_0 - PA) - 0.5 * kappa_b * B
    return np.array([dPA, dB])


def tangent(string):
    T = np.zeros_like(string)
    T[1:-1] = string[2:] - string[:-2]
    T[0] = string[1] - string[0]
    T[-1] = string[-1] - string[-2]
    norms = np.linalg.norm(T, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return T / norms

def reparametrize(string):
    # arc-length parametrization
    ds = np.linalg.norm(np.diff(string, axis=0), axis=1)
    s = np.zeros(len(string))
    s[1:] = np.cumsum(ds)

    if s[-1] == 0:
        return string.copy()

    s_uniform = np.linspace(0, s[-1], len(string))

    PA_interp = interp1d(s, string[:, 0], kind='linear')
    B_interp = interp1d(s, string[:, 1], kind='linear')

    new_string = np.zeros_like(string)
    new_string[:, 0] = PA_interp(s_uniform)
    new_string[:, 1] = B_interp(s_uniform)
    return new_string

def initialize_string(N):
    # better than a straight line: use adiabatic curve
    PA_vals = np.linspace(0.0, A_0, N)
    B_vals = (2*np.sqrt(2)*g/kappa_b) * PA_vals * (A_0 - PA_vals)
    return np.column_stack([PA_vals, B_vals])

def string_step(string, dtau):
    new_string = string.copy()
    T = tangent(string)

    for i in range(1, len(string)-1):
        Fi = F(string[i])

        # normal component only
        Fi_normal = Fi - np.dot(Fi, T[i]) * T[i]

        new_string[i] += dtau * Fi_normal

    # fix endpoints
    new_string[0] = np.array([0.0, 0.0])
    new_string[-1] = np.array([A_0, 0.0])

    # redistribute images
    new_string = reparametrize(new_string)

    # fix endpoints again after interpolation
    new_string[0] = np.array([0.0, 0.0])
    new_string[-1] = np.array([A_0, 0.0])

    return new_string

def string_method(N=200, dtau=1e-3, max_iter=20000, tol=1e-8):
    string = initialize_string(N)

    for k in range(max_iter):
        old = string.copy()
        string = string_step(string, dtau)

        err = np.max(np.linalg.norm(string - old, axis=1))
        if k % 200 == 0:
            print(f"iter={k}, err={err:.3e}")

        if err < tol:
            print(f"converged at iter={k}, err={err:.3e}")
            break

    return string

In [ ]:
N=200
string_init = initialize_string(N)
string_test = string_method(dtau=1e-2)

In [ ]:
plt.plot(string_test[:,0],string_test[:,1])
plt.plot(string_init[:,0],string_init[:,1])
plt.xlabel(r'$P_A$')
plt.ylabel(r'$B$')
plt.show()

In [ ]:
string_1 = []
kappa_b_list = np.linspace(1,5,5)/10.4
for kappa_b in kappa_b_list:
    string_1.append(string_method(dtau=1e-2))

## Compute action

Action $S = \int_0^1\langle \phi',\theta \rangle ds = \int_0^1A'P_A+B'P_B ds=\int_0^1A'(A_0-A)-B'Bds=-A_0^2/2$

# Second part

In [ ]:
import numpy as np
from scipy.special import jv

INF = np.inf

hbar = 1
e = 1
#beta = INF  # Zero-temperature case: T = 0, beta -> inf
phi_quantum = 1 / 2

# Circuit parameters
omega_a = 25.338776456203686
omega_b = 2 * omega_a

phi_a = 0.11
phi_b = 0.204

E_J = 37.12 * 2 * np.pi

# Environment
kappa_b = 5 / 10.4  # 1/ns from Leon's paper

omega_d = omega_b                  # Drive frequency
omega_p = 2 * omega_a - omega_d   # Pump frequency

epsilon_p = 0.1
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b  # Leon's paper
epsilon_d = 4 * g2

g = E_J * np.sin(epsilon_p) * phi_a**2 * phi_b
epsilon_d_ratio = 2
params = (epsilon_p, E_J, epsilon_d_ratio, kappa_b)

M = 100

A_0 = np.sqrt(epsilon_d / g)
B_0 = np.sqrt(2) * epsilon_d / kappa_b

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp

# parameters
u0 = A_0**2

alpha = 1/4
beta  = epsilon_d/(4*g)
gamma = kappa_b/(4*np.sqrt(2)*g)

# local slopes at u = u0
disc = np.sqrt(gamma**2 + 4*u0*alpha)
m_plus  = (-gamma + disc) / (2*u0)
m_minus = (-gamma - disc) / (2*u0)

# choose the branch connected to the adiabatic one
m = m_plus

eps = 1e-6
x_init = u0 - eps
Psi_init = -m * eps
x_fin = 1e-6

def rhs(x, y):
    Psi = y[0]
    return [(alpha*x - beta - gamma*Psi)/(x*Psi)]

sol = solve_ivp(
    rhs,
    t_span=(x_init, x_fin),
    y0=[Psi_init],
    dense_output=True,
    max_step=0.01,
    rtol=1e-8,
    atol=1e-10
)

print(sol.message)

In [ ]:
import matplotlib.pyplot as plt
xx = np.linspace(x_init, x_fin, 1000)
yy = sol.sol(xx)[0]

P_A_test_sq= np.linspace(0,A_0**2,100)
B_test = np.sqrt(2)*(g*P_A_test_sq-epsilon_d)/kappa_b

plt.plot(xx, yy)
plt.plot(P_A_test_sq, B_test)
plt.xlabel("x")
plt.ylabel(r"$\phi(x)$")
#plt.xlim(1.4, 1.42)
#plt.ylim(-0.1, 0.1)
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
from scipy.interpolate import interp1d

# parameters
# assume g, epsilon_d, kappa_b already defined
A_0 = np.sqrt(epsilon_d / g)
B_0 = np.sqrt(2) * epsilon_d / kappa_b

# reduced vector field X = (u, B), where u = P_A^2
def F(X):
    u, B = X
    du = 2*np.sqrt(2) * g * u * B
    dB = -epsilon_d/np.sqrt(2) + g*u/np.sqrt(2) - 0.5*kappa_b*B
    return np.array([du, dB])

def tangent(string):
    T = np.zeros_like(string)
    T[1:-1] = string[2:] - string[:-2]
    T[0] = string[1] - string[0]
    T[-1] = string[-1] - string[-2]
    norms = np.linalg.norm(T, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return T / norms

def reparametrize(string):
    ds = np.linalg.norm(np.diff(string, axis=0), axis=1)
    s = np.zeros(len(string))
    s[1:] = np.cumsum(ds)

    if s[-1] == 0:
        return string.copy()

    s_uniform = np.linspace(0, s[-1], len(string))

    u_interp = interp1d(s, string[:, 0], kind='linear')
    B_interp = interp1d(s, string[:, 1], kind='linear')

    new_string = np.zeros_like(string)
    new_string[:, 0] = u_interp(s_uniform)
    new_string[:, 1] = B_interp(s_uniform)
    return new_string

def initialize_string(N):
    # adiabatic curve: B_ad(u) = sqrt(2)/kappa_b * (g*u - epsilon_d)
    u_vals = np.linspace(A_0**2, 0.0, N)
    B_vals = (np.sqrt(2)/kappa_b) * (g*u_vals - epsilon_d)
    return np.column_stack([u_vals, B_vals])

def string_step(string, dtau):
    new_string = string.copy()
    T = tangent(string)

    for i in range(1, len(string)-1):
        Fi = F(string[i])

        # normal component only
        Fi_normal = Fi - np.dot(Fi, T[i]) * T[i]

        # relaxation sign
        new_string[i] += dtau * Fi_normal

    # fix endpoints
    new_string[0]  = np.array([A_0**2, 0.0])
    new_string[-1] = np.array([0.0, -B_0])

    # redistribute images
    new_string = reparametrize(new_string)

    # fix endpoints again after interpolation
    new_string[0]  = np.array([A_0**2, 0.0])
    new_string[-1] = np.array([0.0, -B_0])

    return new_string

def residual_norm(string):
    T = tangent(string)
    vals = []
    for i in range(1, len(string)-1):
        Fi = F(string[i])
        Fi_normal = Fi - np.dot(Fi, T[i]) * T[i]
        vals.append(np.linalg.norm(Fi_normal))
    return max(vals) if vals else 0.0

def string_method(N=200, dtau=1e-3, max_iter=20000, tol=1e-8):
    string = initialize_string(N)

    for k in range(max_iter):
        old = string.copy()
        string = string_step(string, dtau)

        err = np.max(np.linalg.norm(string - old, axis=1))
        res = residual_norm(string)

        if k % 200 == 0:
            print(f"iter={k}, move={err:.3e}, res={res:.3e}")

        if err < tol and res < tol:
            print(f"converged at iter={k}, move={err:.3e}, res={res:.3e}")
            break

    return string

In [ ]:
N=200
string_init = initialize_string(N)
string_test = string_method(dtau=1e-2)

In [ ]:
plt.plot(string_test[:,0],string_test[:,1])
plt.plot(string_init[:,0],string_init[:,1])

In [ ]:
string_2 = []
kappa_b_list = np.linspace(1,5,5)/10.4
for kappa_b in kappa_b_list:
    B_0 = np.sqrt(2) * epsilon_d / kappa_b
    string_2.append(string_method(dtau=1e-2))

## Compute the action

$S = 0$

# Whole path

In [ ]:
for (i,kappa_b) in enumerate(kappa_b_list):
    ax = plt.figure().add_subplot()
    string_1_i = string_1[i]
    P_A_1 = string_1_i[:,0]
    B_1 = string_1_i[:,1]
    string_2_i = string_2[i]
    P_A_2 = np.sqrt(string_2_i[:,0])
    B_2 = string_2_i[:,1]

    ax.plot(P_A_1, B_1)

    ax.plot(P_A_2, B_2)

    ax.set_xlabel(r'$P_A$')
    ax.set_ylabel(r'$B$')
    ax.set_title(r"$\kappa_b=$" + str(round(kappa_b, 1)) + " GHz, " + r"$\frac{\kappa_b}{4\sqrt{2}g\alpha}=$"+str(round(kappa_b/4/np.sqrt(2)/g/A_0, 2)))
    plt.tight_layout()
    plt.show()

In [ ]:
from Plot_setting import Perso_colormap
v_ext = 0.25
for (i,kappa_b) in enumerate(kappa_b_list):
    ax = plt.figure().add_subplot(projection='3d')
    string_1_i = string_1[i]
    P_A_1 = string_1_i[:,0]
    B_1 = string_1_i[:,1]
    A_1 = A_0-string_1_i[:,0]
    P_B_1 = -B_1
    string_2_i = string_2[i]
    P_A_2 = np.sqrt(string_2_i[:,0])
    B_2 = string_2_i[:,1]
    A_2 = np.zeros(len(string_2_i[:,0]))
    P_B_2 = A_2

    sc = ax.scatter(P_A_1, B_1, A_1,
                    c=P_B_1,
                    cmap=Perso_colormap,
                    vmin=-v_ext, vmax=v_ext)

    ax.scatter(P_A_2, B_2, A_2,
            c=P_B_2,
            cmap=Perso_colormap,
            vmin=-v_ext, vmax=v_ext)

    ax.set_xlabel(r'$P_A$')
    ax.set_ylabel(r'$B$')
    ax.set_zlabel(r'$A$')
    ax.set_title(r"$\kappa_b=$" + str(round(kappa_b, 1)) + " GHz")

    plt.colorbar(sc, ax=ax, label=r"$P_B$", pad=0.25)
    plt.tight_layout()
    plt.show()

# String on the whole path

In [ ]:
import numpy as np
from scipy.special import jv

INF = np.inf

hbar = 1
e = 1
#beta = INF  # Zero-temperature case: T = 0, beta -> inf
phi_quantum = 1 / 2

# Circuit parameters
omega_a = 25.338776456203686
omega_b = 2 * omega_a

phi_a = 0.11
phi_b = 0.204

E_J = 37.12 * 2 * np.pi

# Environment
kappa_b = 5 / 10.4  # 1/ns from Leon's paper

omega_d = omega_b                  # Drive frequency
omega_p = 2 * omega_a - omega_d   # Pump frequency

epsilon_p = 0.1
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b  # Leon's paper
epsilon_d = 4 * g2

g = E_J * np.sin(epsilon_p) * phi_a**2 * phi_b
epsilon_d_ratio = 2
params = (epsilon_p, E_J, epsilon_d_ratio, kappa_b)

M = 100

A_0 = np.sqrt(epsilon_d / g)
B_0 = np.sqrt(2) * epsilon_d / kappa_b

In [ ]:
def F(X):
    A, P_A, B, P_B = X
    dA = -np.sqrt(2)*g*(A*B-P_A*P_B)
    dP_A = -np.sqrt(2)*g*(A*P_B-P_A*B)
    dB = -epsilon_d/np.sqrt(2) + g*(A**2+P_A**2)/np.sqrt(2) - kappa_b*(B/2+P_B)
    dP_B = np.sqrt(2)*g*A*P_A+kappa_b*P_B/2
    return np.array([dA, dP_A, dB, dP_B])

def Hamiltonian(X):
    A, P_A, B, P_B = X
    return -np.sqrt(2)*g*A*P_A*B + (g*(A**2 + P_A**2)/np.sqrt(2) - epsilon_d/np.sqrt(2) - kappa_b*B/2)*P_B - kappa_b*P_B**2/2

def tangent(string):
    T = np.zeros_like(string)
    T[1:-1] = string[2:] - string[:-2]
    T[0] = string[1] - string[0]
    T[-1] = string[-1] - string[-2]
    norms = np.linalg.norm(T, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return T / norms

def reparametrize(string):
    ds = np.linalg.norm(np.diff(string, axis=0), axis=1)
    s = np.zeros(len(string))
    s[1:] = np.cumsum(ds)

    if s[-1] == 0:
        return string.copy()

    s_uniform = np.linspace(0, s[-1], len(string))

    A_interp = interp1d(s, string[:, 0], kind='linear')
    P_A_interp = interp1d(s, string[:, 1], kind='linear')
    B_interp = interp1d(s, string[:, 2], kind='linear')
    P_B_interp = interp1d(s, string[:, 3], kind='linear')

    new_string = np.zeros_like(string)
    new_string[:, 0] = A_interp(s_uniform)
    new_string[:, 1] = P_A_interp(s_uniform)
    new_string[:, 2] = B_interp(s_uniform)
    new_string[:, 3] = P_B_interp(s_uniform)
    return new_string

def initialize_string(N):
    P_A_part_1 = string_1[-1][:,0]#np.linspace(0.0, A_0, N)
    P_A_part_2 = np.sqrt(string_2[-1][:,0])#np.linspace(A_0, 0.0, N)
    B_part_1 =  string_1[-1][:,1]#(2*np.sqrt(2)*g/kappa_b) * P_A_part_1 * (A_0 - P_A_part_1)
    B_part_2 = string_2[-1][:,1]#(np.sqrt(2)/kappa_b) * (g*P_A_part_2**2 - epsilon_d)
    A_vals = np.concatenate((A_0-P_A_part_1, np.zeros(N)))
    P_A_vals = np.concatenate((P_A_part_1, P_A_part_2))
    B_vals = np.concatenate((B_part_1, B_part_2))
    P_B_vals = np.concatenate((-B_part_1, np.zeros(N)))
    return np.column_stack([A_vals, P_A_vals, B_vals, P_B_vals])

def string_step(string, dtau):
    new_string = string.copy()
    T = tangent(string)

    for i in range(1, len(string)-1):
        Xi = string[i]
        Fi = F(Xi)

        # normal component only
        Fi_normal = Fi - np.dot(Fi, T[i]) * T[i]

        # relaxation step: use the stable sign
        new_string[i] = Xi + dtau * np.array([-Fi_normal[0],Fi_normal[1],Fi_normal[2],-Fi_normal[3]])

        # project updated point back to H = 0
        Xi_proj = new_string[i]
        Fi_proj = F(Xi_proj)
        grad_H = np.array([-Fi_proj[1], Fi_proj[0], -Fi_proj[3], Fi_proj[2]])
        grad_norm_sq = np.dot(grad_H, grad_H)

        if grad_norm_sq > 1e-14:
            Hi = Hamiltonian(Xi_proj)
            new_string[i] -= Hi * grad_H / grad_norm_sq

    # fix endpoints
    new_string[0:10]  = string[0:10]
    new_string[-11:-1] = string[-11:-1]

    # redistribute images
    new_string = reparametrize(new_string)

    # fix endpoints again after interpolation
    new_string[0:10]  = string[0:10]
    new_string[-11:-1] = string[-11:-1]

    return new_string

def residual_norm(string):
    T = tangent(string)
    vals = []
    for i in range(1, len(string)-1):
        Fi = F(string[i])
        Fi_normal = Fi - np.dot(Fi, T[i]) * T[i]
        vals.append(np.linalg.norm(Fi_normal))
    return max(vals) if vals else 0.0

def string_method(N=200, dtau=1e-3, max_iter=20000, tol=1e-8):
    res_list = []
    move_list = []
    string = initialize_string(N)

    for k in range(max_iter):
        old = string.copy()
        string = string_step(string, dtau)
        move = np.max(np.linalg.norm(string - old, axis=1))
        res = residual_norm(string)
        res_list.append(res)
        move_list.append(move)

        if k % 200 == 0:
            print(f"iter={k}, res={res:.3e}")

        if res < tol:
            print(f"converged at iter={k}, res={res:.3e}")
            break

    return string, res_list, move_list

In [ ]:
N=200
string_init = initialize_string(N)
string_test, res_list, move_list = string_method(dtau=.2, max_iter=5000)

In [ ]:
fig, ax1 = plt.subplots()

# First y-axis
ax1.plot(res_list, label="Residuals", color="C0")
ax1.set_ylabel("Residuals", color="C0")
ax1.set_xlabel("Optimization step")
ax1.tick_params(axis='y', labelcolor="C0")

# Second y-axis
ax2 = ax1.twinx()
ax2.plot(move_list, label="Move", color="C1")
ax2.set_ylabel("Move", color="C1")
ax2.tick_params(axis='y', labelcolor="C1")

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(string_init[:,0],string_init[:,1])
plt.plot(string_test[:,0],string_test[:,1])
plt.xlabel(r'$A$')
plt.ylabel(r'$P_A$')
plt.show()

In [ ]:
plt.plot(string_init[:,2],string_init[:,3])
plt.plot(string_test[:,2],string_test[:,3])
plt.xlabel(r'$B$')
plt.ylabel(r'$P_B$')
plt.show()

In [ ]:
H_string=[]
for X in string_test:
    H_string.append(Hamiltonian(X))

plt.plot(abs(np.array(H_string)))
plt.yaxis("log")

In [ ]:
def action_from_string(string):
    A  = string[:, 0]
    PA = string[:, 1]
    B  = string[:, 2]
    PB = string[:, 3]

    dA = np.diff(A)
    dB = np.diff(B)

    PA_mid = 0.5 * (PA[:-1] + PA[1:])
    PB_mid = 0.5 * (PB[:-1] + PB[1:])

    S = np.sum(PA_mid * dA + PB_mid * dB)
    return S

In [ ]:
action_from_string(string_test)

In [ ]:
action_from_string(string_init)